# Find unlabeled clips and optionally delete them

This notebook compares local `data/clips/**/*.mp4` files against the clip URIs referenced in `data/labels.jsonl`, lists any clips that do not have a matching label entry, and deletes them only when `DELETE_UNLABELED = True`.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Video, display

from aitraf.data_ops.schema import LabelsSchema
from aitraf.data_ops.utils import strip_clips_prefix
from aitraf.utils.s3_utils import parse_s3_uri

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
CLIPS_DIR = DATA_DIR / "clips"
LABELS_PATH = DATA_DIR / "labels.jsonl"

DELETE_UNLABELED = False
PREVIEW_COUNT = 5

pd.set_option("display.max_colwidth", 140)


In [9]:
def clip_path_to_relative_key(clip_path: Path) -> str:
    return clip_path.relative_to(CLIPS_DIR).as_posix()


def s3_uri_to_relative_key(video_uri: str) -> str:
    _, key = parse_s3_uri(video_uri)
    return strip_clips_prefix(Path(key)).as_posix()


clip_paths = sorted(CLIPS_DIR.rglob("*.mp4"))
if not clip_paths:
    raise FileNotFoundError(f"No clips found under {CLIPS_DIR}")

if not LABELS_PATH.exists():
    raise FileNotFoundError(f"Labels file not found at {LABELS_PATH}")

clips_df = pd.DataFrame({"clip_path": clip_paths})
clips_df["clip_key"] = clips_df["clip_path"].map(clip_path_to_relative_key)
clips_df["clip_name"] = clips_df["clip_path"].map(lambda path: path.name)
clips_df["clip_relpath"] = clips_df["clip_path"].map(
    lambda path: path.relative_to(PROJECT_ROOT).as_posix()
)

labels_df = pd.read_json(LABELS_PATH, lines=True, dtype=LabelsSchema.types)
labels_df["clip_key"] = labels_df["video"].astype(str).map(s3_uri_to_relative_key)
labels_df["label_count"] = 1

label_summary_df = (
    labels_df.groupby("clip_key", as_index=False)
    .agg(
        label_count=("label_count", "sum"),
        video=("video", "first"),
        trick=("trick", "first"),
        person=("person", "first"),
        key_foot=("key_foot", "first"),
        execution_score=("execution_score", "first"),
    )
)

inventory_df = clips_df.merge(label_summary_df, on="clip_key", how="left")
inventory_df["label_count"] = inventory_df["label_count"].fillna(0).astype(int)
inventory_df["has_label"] = inventory_df["label_count"] > 0
inventory_df = inventory_df.sort_values(["has_label", "clip_key"]).reset_index(drop=True)

inventory_df.head()


clip_path  \
0  /workspace/data/clips/25-11-13 20-29-29 5726-00.00.07.483-00.00.15.078-seg1.mp4   
1  /workspace/data/clips/25-11-13 20-29-29 5726-00.01.15.659-00.01.20.388-seg2.mp4   
2  /workspace/data/clips/25-11-13 20-29-29 5726-00.01.42.372-00.01.48.259-seg3.mp4   
3  /workspace/data/clips/25-11-13 20-29-29 5726-00.01.55.402-00.01.59.652-seg4.mp4   
4  /workspace/data/clips/25-11-13 20-29-29 5726-00.02.25.021-00.02.31.951-seg5.mp4   

                                                    clip_key  \
0  25-11-13 20-29-29 5726-00.00.07.483-00.00.15.078-seg1.mp4   
1  25-11-13 20-29-29 5726-00.01.15.659-00.01.20.388-seg2.mp4   
2  25-11-13 20-29-29 5726-00.01.42.372-00.01.48.259-seg3.mp4   
3  25-11-13 20-29-29 5726-00.01.55.402-00.01.59.652-seg4.mp4   
4  25-11-13 20-29-29 5726-00.02.25.021-00.02.31.951-seg5.mp4   

                                                   clip_name  \
0  25-11-13 20-29-29 5726-00.00.07.483-00.00.15.078-seg1.mp4   
1  25-11-13 20-29-29 5726-00.01.15.659-00.01.20.388-seg2.mp4   
2  25-11-13 20-29-29 5726-00.01.42.372-00.01.48.259-seg3.mp4   
3  25-11-13 20-29-29 5726-00.01.55.402-00.01.59.652-seg4.mp4   
4  25-11-13 20-29-29 5726-00.02.25.021-00.02.31.951-seg5.mp4   

                                                           clip_relpath  \
0  data/clips/25-11-13 20-29-29 5726-00.00.07.483-00.00.15.078-seg1.mp4   
1  data/clips/25-11-13 20-29-29 5726-00.01.15.659-00.01.20.388-seg2.mp4   
2  data/clips/25-11-13 20-29-29 5726-00.01.42.372-00.01.48.259-seg3.mp4   
3  data/clips/25-11-13 20-29-29 5726-00.01.55.402-00.01.59.652-seg4.mp4   
4  data/clips/25-11-13 20-29-29 5726-00.02.25.021-00.02.31.951-seg5.mp4   

   label_count video trick person key_foot  execution_score  \
0            0  <NA>  <NA>   <NA>     <NA>             <NA>   
1            0  <NA>  <NA>   <NA>     <NA>             <NA>   
2            0  <NA>  <NA>   <NA>     <NA>             <NA>   
3            0  <NA>  <NA>   <NA>     <NA>             <NA>   
4            0  <NA>  <NA>   <NA>     <NA>             <NA>   

0                  <NA>      False  
1                  <NA>      False  
2                  <NA>      False  
3                  <NA>      False  
4                  <NA>      False

In [10]:
unlabeled_df = inventory_df.loc[~inventory_df["has_label"]].copy()
clip_keys_on_disk = set(clips_df["clip_key"])
labeled_missing_on_disk_df = label_summary_df.loc[
    ~label_summary_df["clip_key"].isin(clip_keys_on_disk)
].copy()
duplicate_label_rows = int(labels_df["clip_key"].duplicated().sum())

summary_df = pd.DataFrame(
    [
        {"metric": "clip_files_on_disk", "value": len(clips_df)},
        {
            "metric": "clip_files_with_labels_on_disk",
            "value": int(inventory_df["has_label"].sum()),
        },
        {"metric": "unlabeled_clip_files_on_disk", "value": len(unlabeled_df)},
        {"metric": "label_rows", "value": len(labels_df)},
        {"metric": "unique_labeled_clip_keys", "value": len(label_summary_df)},
        {
            "metric": "duplicate_label_rows_for_same_clip",
            "value": duplicate_label_rows,
        },
        {
            "metric": "labeled_clip_keys_missing_on_disk",
            "value": len(labeled_missing_on_disk_df),
        },
    ]
)

display(summary_df)

if unlabeled_df.empty:
    print("No unlabeled clips found.")
else:
    display(unlabeled_df[["clip_key", "clip_relpath"]].reset_index(drop=True))


,metric,value
0,clip_files_on_disk,704
1,clip_files_with_labels_on_disk,689
2,unlabeled_clip_files_on_disk,15
3,label_rows,689
4,unique_labeled_clip_keys,689
5,duplicate_label_rows_for_same_clip,0
6,labeled_clip_keys_missing_on_disk,0


,clip_key,clip_relpath
0,25-11-13 20-29-29 5726-00.00.07.483-00.00.15.078-seg1.mp4,data/clips/25-11-13 20-29-29 5726-00.00.07.483-00.00.15.078-seg1.mp4
1,25-11-13 20-29-29 5726-00.01.15.659-00.01.20.388-seg2.mp4,data/clips/25-11-13 20-29-29 5726-00.01.15.659-00.01.20.388-seg2.mp4
2,25-11-13 20-29-29 5726-00.01.42.372-00.01.48.259-seg3.mp4,data/clips/25-11-13 20-29-29 5726-00.01.42.372-00.01.48.259-seg3.mp4
3,25-11-13 20-29-29 5726-00.01.55.402-00.01.59.652-seg4.mp4,data/clips/25-11-13 20-29-29 5726-00.01.55.402-00.01.59.652-seg4.mp4
4,25-11-13 20-29-29 5726-00.02.25.021-00.02.31.951-seg5.mp4,data/clips/25-11-13 20-29-29 5726-00.02.25.021-00.02.31.951-seg5.mp4
5,25-11-13 20-29-29 5726-00.03.29.072-00.03.35.634-seg6.mp4,data/clips/25-11-13 20-29-29 5726-00.03.29.072-00.03.35.634-seg6.mp4
6,25-11-13 20-29-29 5726-00.04.14.816-00.04.21.407-seg7.mp4,data/clips/25-11-13 20-29-29 5726-00.04.14.816-00.04.21.407-seg7.mp4
7,IMG_5682-00.00.41.689-00.00.49.035-seg1.mp4,data/clips/IMG_5682-00.00.41.689-00.00.49.035-seg1.mp4
8,IMG_5682-00.01.08.753-00.01.14.367-seg2.mp4,data/clips/IMG_5682-00.01.08.753-00.01.14.367-seg2.mp4
9,IMG_5682-00.02.24.967-00.02.30.080-seg3.mp4,data/clips/IMG_5682-00.02.24.967-00.02.30.080-seg3.mp4


In [11]:
preview_df = unlabeled_df.head(PREVIEW_COUNT).copy()

if preview_df.empty:
    print("No unlabeled clips to preview.")
else:
    display(preview_df[["clip_key", "clip_relpath"]])
    for row in preview_df.itertuples(index=False):
        print(row.clip_key)
        display(Video(str(row.clip_path), embed=False, width=320))


,clip_key,clip_relpath
0,25-11-13 20-29-29 5726-00.00.07.483-00.00.15.078-seg1.mp4,data/clips/25-11-13 20-29-29 5726-00.00.07.483-00.00.15.078-seg1.mp4
1,25-11-13 20-29-29 5726-00.01.15.659-00.01.20.388-seg2.mp4,data/clips/25-11-13 20-29-29 5726-00.01.15.659-00.01.20.388-seg2.mp4
2,25-11-13 20-29-29 5726-00.01.42.372-00.01.48.259-seg3.mp4,data/clips/25-11-13 20-29-29 5726-00.01.42.372-00.01.48.259-seg3.mp4
3,25-11-13 20-29-29 5726-00.01.55.402-00.01.59.652-seg4.mp4,data/clips/25-11-13 20-29-29 5726-00.01.55.402-00.01.59.652-seg4.mp4
4,25-11-13 20-29-29 5726-00.02.25.021-00.02.31.951-seg5.mp4,data/clips/25-11-13 20-29-29 5726-00.02.25.021-00.02.31.951-seg5.mp4


25-11-13 20-29-29 5726-00.00.07.483-00.00.15.078-seg1.mp4


25-11-13 20-29-29 5726-00.01.15.659-00.01.20.388-seg2.mp4


25-11-13 20-29-29 5726-00.01.42.372-00.01.48.259-seg3.mp4


25-11-13 20-29-29 5726-00.01.55.402-00.01.59.652-seg4.mp4


25-11-13 20-29-29 5726-00.02.25.021-00.02.31.951-seg5.mp4


In [12]:
deleted_paths = []

if DELETE_UNLABELED:
    for clip_path in unlabeled_df["clip_path"]:
        if clip_path.exists():
            clip_path.unlink()
            deleted_paths.append(clip_path)

    print(f"Deleted {len(deleted_paths)} unlabeled clips from {CLIPS_DIR}")
else:
    print(
        "Dry run only. Set DELETE_UNLABELED = True and rerun this cell to delete the unlabeled clips listed above."
    )
    print(f"{len(unlabeled_df)} unlabeled clips would be deleted.")

action_paths = deleted_paths if DELETE_UNLABELED else unlabeled_df["clip_path"].tolist()
action_df = pd.DataFrame(
    {
        "clip_relpath": [
            path.relative_to(PROJECT_ROOT).as_posix()
            for path in action_paths
        ]
    }
)

display(action_df)

remaining_clip_count = sum(1 for _ in CLIPS_DIR.rglob("*.mp4"))
print(f"Clips currently on disk: {remaining_clip_count}")


Deleted 15 unlabeled clips from /workspace/data/clips


,clip_relpath
0,data/clips/25-11-13 20-29-29 5726-00.00.07.483-00.00.15.078-seg1.mp4
1,data/clips/25-11-13 20-29-29 5726-00.01.15.659-00.01.20.388-seg2.mp4
2,data/clips/25-11-13 20-29-29 5726-00.01.42.372-00.01.48.259-seg3.mp4
3,data/clips/25-11-13 20-29-29 5726-00.01.55.402-00.01.59.652-seg4.mp4
4,data/clips/25-11-13 20-29-29 5726-00.02.25.021-00.02.31.951-seg5.mp4
5,data/clips/25-11-13 20-29-29 5726-00.03.29.072-00.03.35.634-seg6.mp4
6,data/clips/25-11-13 20-29-29 5726-00.04.14.816-00.04.21.407-seg7.mp4
7,data/clips/IMG_5682-00.00.41.689-00.00.49.035-seg1.mp4
8,data/clips/IMG_5682-00.01.08.753-00.01.14.367-seg2.mp4
9,data/clips/IMG_5682-00.02.24.967-00.02.30.080-seg3.mp4


Clips currently on disk: 689
